In [29]:
#import
import pandas as pd
import numpy as np

df=pd.read_csv("raw_employee_dataset.csv")

df_raw=df.copy()
print("(Row ,Column) :",df.shape)

display(df.head)

(Row ,Column) : (36, 7)


<bound method NDFrame.head of    Employee_ID            Name  Age       City         Salary     Join_Date  \
0         E001    Aarav Sharma   25    Kolkata          45000    2022-01-15   
1         E002        Diya Sen   29   Calcutta          52000    15/03/2021   
2         E003       Rohan Das   31   kolkata           61000  Mar 20, 2020   
3         E004        Mira Roy   27  New Delhi          48000    2023/04/10   
4         E005     Kabir Ghosh   34      Delhi          72000    2020-11-01   
5         E006     Ananya Paul   24     Mumbai            NaN    2024-02-05   
6         E007     Ishaan Bose   41     MUMBAI          98000    05-06-2018   
7         E008      Tara Dutta   30     Bombay          67000    2019-07-21   
8         E009  Neel Mukherjee   28       Pune          56000    2021-12-12   
9         E010       Sana Khan   36      Pune           83000    2020-08-30   
10        E011     Arjun Mitra   26  Bengaluru         59000     2022-09-01   
11        E012  Ira Ch

In [30]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Employee_ID  36 non-null     object
 1   Name         36 non-null     object
 2   Age          36 non-null     object
 3   City         36 non-null     object
 4   Salary       35 non-null     object
 5   Join_Date    36 non-null     object
 6   Department   36 non-null     object
dtypes: object(7)
memory usage: 2.1+ KB
None


In [31]:
#check missing values
print(df.isnull().sum())
print("\n")
#check duplicate rows
print(df.duplicated().sum())


#check city names
print("City")
print(df["City"].value_counts(dropna=False))

#check departments
print("Department")
print(df["Department"].value_counts(dropna=False))

Employee_ID    0
Name           0
Age            0
City           0
Salary         1
Join_Date      0
Department     0
dtype: int64


0
City
City
Kolkata      5
Delhi        4
Mumbai       4
Chennai      3
Bengaluru    2
Jaipur       2
Hydrabad     2
Bangalore    2
Pune         2
New Delhi    2
Ahmedabad    2
Pune         1
Calcutta     1
Bombay       1
MUMBAI       1
Hyderabad    1
kolkata      1
Name: count, dtype: int64
Department
Department
IT         12
Finance     9
Sales       8
HR          7
Name: count, dtype: int64


In [32]:
#Remove Extra Spaces 
for col in df.select_dtypes(include="object").columns:
    df[col]=df[col].str.strip()

#Correct names

city_map={
    "Calcutta":"Kolkata",
    "kolkata": "Kolkata",
    "New Delhi": "Delhi",
    "Bombay": "Mumbai",
    "MUMBAI": "Mumbai",
    "Bangalore": "Bengaluru",
    "Hydrabad": "Hyderabad"
}

df["City"] = df["City"].replace(city_map)

# Standardize capitalization
df["City"] = df["City"].str.title()

print(df["City"].value_counts())

City
Kolkata      7
Delhi        6
Mumbai       6
Bengaluru    4
Pune         3
Chennai      3
Hyderabad    3
Jaipur       2
Ahmedabad    2
Name: count, dtype: int64


In [33]:
# Convert Age into numbers
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

# Convert Salary into numbers
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")

print("Missing ages:", df["Age"].isnull().sum())
print("Missing salaries:", df["Salary"].isnull().sum())

Missing ages: 1
Missing salaries: 2


In [34]:
# Convert different date formats into one standard format
df["Join_Date"] = pd.to_datetime(
    df["Join_Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print(df["Join_Date"].head(10))
print("Invalid or missing dates:", df["Join_Date"].isnull().sum())

0   2022-01-15
1   2021-03-15
2   2020-03-20
3   2023-04-10
4   2020-11-01
5   2024-02-05
6   2018-06-05
7   2019-07-21
8   2021-12-12
9   2020-08-30
Name: Join_Date, dtype: datetime64[ns]
Invalid or missing dates: 1


In [35]:
# Count exact duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())

# Find possible duplicates based on employee details
possible_duplicates = df[
    df.duplicated(
        subset=["Name", "Join_Date", "Department"],
        keep=False
    )
]

display(possible_duplicates)

Exact duplicate rows: 0


,Employee_ID,Name,Age,City,Salary,Join_Date,Department
0,E001,Aarav Sharma,25.0,Kolkata,45000.0,2022-01-15,IT
28,E029,Aarav Sharma,25.0,Kolkata,45000.0,2022-01-15,IT


In [36]:
# Check missing values after conversion
print(df.isnull().sum())
print("Missing values handled without blanket filling.")

Employee_ID    0
Name           0
Age            1
City           0
Salary         2
Join_Date      1
Department     0
dtype: int64
Missing values handled without blanket filling.


In [37]:
# Flag ages outside a plausible range
df["Age_Flag"] = (
    df["Age"].notna()
    & ~df["Age"].between(18, 80)
)

# Flag negative, zero or unusually high salaries
df["Salary_Flag"] = (
    df["Salary"].notna()
    & (
        (df["Salary"] <= 0)
        | (df["Salary"] > 500000)
    )
)

print("Suspicious ages:")
display(
    df.loc[
        df["Age_Flag"],
        ["Employee_ID", "Name", "Age"]
    ]
)

print("Suspicious salaries:")
display(
    df.loc[
        df["Salary_Flag"],
        ["Employee_ID", "Name", "Salary"]
    ]
)

Suspicious ages:


,Employee_ID,Name,Age
29,E030,Rudra Sen,999.0


Suspicious salaries:


,Employee_ID,Name,Salary
14,E015,Vivaan Saha,1800000.0
30,E031,Ishita Das,-5000.0


In [38]:
# Keep all rows in the dataset

# Set implausible ages to missing pending verification
df.loc[df["Age_Flag"], "Age"] = np.nan

# Set non-positive salaries to missing
df.loc[df["Salary"] <= 0, "Salary"] = np.nan

# Keep unusually high salaries for further investigation

print("Invalid values reviewed.")
print("Total rows retained:", len(df))

Invalid values reviewed.
Total rows retained: 36


In [39]:
# Save the cleaned dataset as a CSV file
df.to_csv(
    "cleaned_employee_dataset.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [40]:
# Display the first few records
display(df.head(10))

# Display final dataset information
df.info()

# Display final statistical summary
display(df.describe(include="all"))

print("Data cleaning completed successfully.")

,Employee_ID,Name,Age,City,Salary,Join_Date,Department,Age_Flag,Salary_Flag
0,E001,Aarav Sharma,25.0,Kolkata,45000.0,2022-01-15,IT,False,False
1,E002,Diya Sen,29.0,Kolkata,52000.0,2021-03-15,HR,False,False
2,E003,Rohan Das,31.0,Kolkata,61000.0,2020-03-20,Finance,False,False
3,E004,Mira Roy,27.0,Delhi,48000.0,2023-04-10,IT,False,False
4,E005,Kabir Ghosh,34.0,Delhi,72000.0,2020-11-01,Sales,False,False
5,E006,Ananya Paul,24.0,Mumbai,NaN,2024-02-05,IT,False,False
6,E007,Ishaan Bose,41.0,Mumbai,98000.0,2018-06-05,Finance,False,False
7,E008,Tara Dutta,30.0,Mumbai,67000.0,2019-07-21,HR,False,False
8,E009,Neel Mukherjee,28.0,Pune,56000.0,2021-12-12,Sales,False,False
9,E010,Sana Khan,36.0,Pune,83000.0,2020-08-30,IT,False,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Employee_ID  36 non-null     object        
 1   Name         36 non-null     object        
 2   Age          34 non-null     float64       
 3   City         36 non-null     object        
 4   Salary       33 non-null     float64       
 5   Join_Date    35 non-null     datetime64[ns]
 6   Department   36 non-null     object        
 7   Age_Flag     36 non-null     bool          
 8   Salary_Flag  36 non-null     bool          
dtypes: bool(2), datetime64[ns](1), float64(2), object(4)
memory usage: 2.2+ KB


,Employee_ID,Name,Age,City,Salary,Join_Date,Department,Age_Flag,Salary_Flag
count,36,36,34.000000,36,3.300000e+01,35,36,36,36
unique,36,35,NaN,9,NaN,NaN,4,2,2
top,E001,Aarav Sharma,NaN,Kolkata,NaN,NaN,IT,False,False
freq,1,2,NaN,7,NaN,NaN,12,35,34
mean,NaN,NaN,31.117647,NaN,1.250909e+05,2020-12-19 22:37:42.857142784,NaN,NaN,NaN
min,NaN,NaN,22.000000,NaN,3.900000e+04,2015-05-12 00:00:00,NaN,NaN,NaN
25%,NaN,NaN,27.250000,NaN,5.800000e+04,2019-11-08 00:00:00,NaN,NaN,NaN
50%,NaN,NaN,30.000000,NaN,6.800000e+04,2021-04-16 00:00:00,NaN,NaN,NaN
75%,NaN,NaN,34.000000,NaN,8.900000e+04,2022-05-05 00:00:00,NaN,NaN,NaN
max,NaN,NaN,45.000000,NaN,1.800000e+06,2024-06-01 00:00:00,NaN,NaN,NaN


Data cleaning completed successfully.
